# knee-mri-detect — train on Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then open the 🔑 *Secrets* panel (left sidebar) and add two secrets with *Notebook access* on:
- `KAGGLE_USERNAME` — your Kaggle username
- `KAGGLE_KEY` — from kaggle.com → Settings → API → Create New Token (the `key` field in the downloaded `kaggle.json`)

Checkpoints are written to Google Drive (`MyDrive/knee-mri-detect/models`) so a disconnect doesn't lose finished planes; re-run all cells and it resumes from the next plane.

In [ ]:
# 1. Repo + deps
!git clone -q https://github.com/Akhil-Prasad09/knee-mri-detect.git 2>/dev/null || (cd knee-mri-detect && git pull -q)
%cd knee-mri-detect
!pip install -q -r requirements.txt
!nvidia-smi -L

In [ ]:
# 2. Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/knee-mri-detect/models'
!mkdir -p "$MODELS"
!ls -la "$MODELS"

In [ ]:
# 3. MRNet from Kaggle (key from Colab Secrets, never pasted here)
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
if not os.path.exists('data/raw/MRNet-v1.0/valid-acl.csv'):
    !kaggle datasets download cjinny/mrnet-v1 -p data/raw --unzip
!ls data/raw/MRNet-v1.0/train/sagittal | wc -l   # expect 1130

Trains sagittal → coronal → axial with `ml/training/config.yaml` (EfficientNet-B3, 20 epochs, ~45–60 min/plane on T4). The best validation-AUC checkpoint per plane is kept. Planes with a `.pt` already on Drive are skipped.

In [ ]:
# 4. Train (resumable per plane)
import yaml, os
cfg = yaml.safe_load(open('ml/training/config.yaml')); cfg['out_dir'] = MODELS
yaml.safe_dump(cfg, open('colab.yaml', 'w'))
for plane in cfg['planes']:
    if os.path.exists(f'{MODELS}/{plane}.pt'):
        print('skip', plane, '(done)'); continue
    !python -m ml.training.train --config colab.yaml --plane $plane
!ls -la "$MODELS"

In [ ]:
# 5. Evaluate: per-plane + ensemble AUC, tuned thresholds -> eval.json
!python -m ml.training.evaluate --config colab.yaml
!cat "$MODELS/eval.json"

`models.zip` lands in Drive at `MyDrive/knee-mri-detect/models.zip`. On the laptop: unzip into `ml/models/`, then `make api`.

In [ ]:
# 6. Bundle for the laptop
!cd "$MODELS" && zip -j ../models.zip *.pt *.json && ls -la ../models.zip